In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/phobertv2-vsmec-2025/other/default/1/model_baseline_2025.pth
/kaggle/input/kg-ekman6/ekman6_4_5.rdf
/kaggle/input/kg-ekman6/ekman6_4_7.rdf
/kaggle/input/kg-ekman6/ekman6_4_4.rdf
/kaggle/input/kg-ekman6/ekman6_4_6.rdf
/kaggle/input/kg-ekman6/ekman6_4_8.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc_2.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc_4.rdf
/kaggle/input/kgvsfcekman/ekman6_2_vsfc_3.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc_weighted.rdf
/kaggle/input/kgvsfcekman/ekman6_4_4.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc_3.rdf
/kaggle/input/kgvsfcekman/ekman6_4_6.rdf
/kaggle/input/kgvsfcekman/ekman6_3_vsfc_5.rdf
/kaggle/input/kgvsfcekman/ekman6_3_vsfc_4.rdf
/kaggle/input/kgvsfcekman/ekman6_3_vsfc_1.rdf
/kaggle/input/kgvsfcekman/ekman6_4_3.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc.rdf
/kaggle/input/uit-vsmec/test_nor_811.xlsx
/kaggle/input/uit-vsmec/train_nor_811.xlsx
/kaggle/input/uit-vsmec/valid_nor_811.xlsx


In [2]:
# ---------------------------------------------------------
# CELL 1: SETUP & DEPENDENCIES
# ---------------------------------------------------------
!pip install -q pyvi rdflib unidecode openpyxl

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import copy
import re
import rdflib
from rdflib import Graph, Namespace, RDF, RDFS, OWL
from pyvi import ViTokenizer
from unidecode import unidecode
import os
import random

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 90.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.8 MB/s eta 0:00:00
✅ Using device: cuda


In [3]:
# =========================================================
# CELL 2: ONTOLOGY ENGINE V12.2 (VSMEC EDITION)
# Logic: Deep Inference (Suy diễn sâu) + Soft Polarity.
# Target: Tối ưu cho 7 nhãn cảm xúc của VSMEC (UIT-VSMEC).
# Input: ekman6_4_8.rdf (File global fix)
# =========================================================
!pip install -q pyvi rdflib unidecode

import os, re
import numpy as np
import rdflib
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, RDFS, OWL
from pyvi import ViTokenizer
from unidecode import unidecode

class OntologyEngine:
    def __init__(self, rdfpath, defaultconf=0.85, manual_boost=1.2, negation_attenuation=0.5, alpha_similarity=0.2, verbose=True):
        self.EKMAN = Namespace("http://example.org/ekman-ontology#")
        self.defaultconf = float(defaultconf)
        self.manual_boost = float(manual_boost)
        self.negation_attenuation = float(negation_attenuation) 
        self.alpha_similarity = float(alpha_similarity)
        self.verbose = bool(verbose)

        # 1. CONFIGS: 7 EMOTIONS (Chuẩn VSMEC/Ekman)
        # Lưu ý: VSMEC dùng 'Enjoyment', ta map về 'Happiness' trong Ontology
        self.ALLEMOTIONS = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
        
        self.ALLAPPRAISALS = [
            "GoalObstructionAppraisal", "PleasantnessAppraisal", "UnpleasantnessAppraisal", 
            "DangerousnessAppraisal", "LossAppraisal", "SuddennessAppraisal", 
            "LowPredictabilityAppraisal", "UnpredictabilityAppraisal", "UnfairnessAppraisal"
        ]
        self.INTENSITY_CLASSES = ["HighIntensity", "MediumIntensity", "LowIntensity"]
        self.POLARITY_CLASSES = ["PositivePolarity", "NegativePolarity", "NeutralPolarity"]

        # 2. MAPPING VSMEC -> ONTOLOGY
        self.EMOTIONMAPPING = {
            "Enjoyment": "Happiness",   # VSMEC label -> Ontology
            "Happiness": "Happiness", 
            "Anger": "Anger", 
            "Disgust": "Disgust", 
            "Fear": "Fear", 
            "Sadness": "Sadness", 
            "Surprise": "Surprise", 
            "Other": "Neutral",         # VSMEC label -> Ontology
            "Neutral": "Neutral", 
            "NeutralEmotion": "Neutral",
            "NegationCue": "NegationCue"
        }
        
        # 3. DEEP INFERENCE RULES (Suy diễn Polarity từ ngữ nghĩa)
        self.IMPLIED_POLARITY = {
            # Emotion -> Polarity
            "Anger": "NegativePolarity", "Disgust": "NegativePolarity",
            "Fear": "NegativePolarity", "Sadness": "NegativePolarity",
            "Happiness": "PositivePolarity", "Enjoyment": "PositivePolarity",
            "Surprise": "PositivePolarity", # Surprise trong VSMEC thường nghiêng về Positive/Neutral
            # Appraisal -> Polarity (Quan trọng để bắt đặc trưng sâu)
            "UnpleasantnessAppraisal": "NegativePolarity",
            "GoalObstructionAppraisal": "NegativePolarity", # Bị cản trở -> Tức giận
            "LossAppraisal": "NegativePolarity",            # Mất mát -> Buồn
            "UnfairnessAppraisal": "NegativePolarity",      # Bất công -> Tức giận
            "DangerousnessAppraisal": "NegativePolarity",   # Nguy hiểm -> Sợ
            "PleasantnessAppraisal": "PositivePolarity"     # Dễ chịu -> Vui
        }
        
        self.AMBIGUOUS_TERMS = {
            "hay", "chả", "cơ", "ngờ", "ý", "quá", "lại", 
            "tôi", "tao", "tớ", "mình", "bạn", "nó", "hắn", "anh", "chị", "em",
            "là", "thì", "mà", "bị", "được", "cái", "con", "người"
        }
        
        self.VALID_DOMAINS = {"GeneralDomain", "SocialDomain", "EducationDomain"}
        
        # LOAD KG
        if self.verbose: print(f"📥 Loading KG for VSMEC: {rdfpath}")
        self.g = Graph()
        if os.path.exists(rdfpath):
            self.g.parse(rdfpath)
            if self.verbose: print(f"   ✅ Loaded {len(self.g)} triples.")
        else:
            print(f"   ⚠️ ERROR: File not found {rdfpath}")

        self.sim_matrix = self._build_similarity_matrix()
        self.lexiconmap = self._build_strict_lexicon()
        
        self.mwe_max_len = 1
        for k in self.lexiconmap.keys(): self.mwe_max_len = max(self.mwe_max_len, len(k.split()))
        if self.verbose: print(f"   📚 Lexicon size: {len(self.lexiconmap)} entries (v12.2 VSMEC-Ready)")

    def _norm(self, s): return re.sub(r"\s+", " ", str(s).lower().strip().replace("_", " ")).strip() if s else ""
    def _local(self, uri): return str(uri).split("#")[-1] if "#" in str(uri) else str(uri).split("/")[-1]

    def _build_similarity_matrix(self):
        n = len(self.ALLEMOTIONS)
        mat = np.zeros((n, n), dtype=np.float32); np.fill_diagonal(mat, 1.0)
        try:
            q = """PREFIX ekman: <http://example.org/ekman-ontology#> SELECT ?score ?e1 ?e2 WHERE { ?edge a ekman:EmotionSimilarityEdge ; ekman:similarityScore ?score ; ekman:linksEmotion ?e1 ; ekman:linksEmotion ?e2 . }"""
            for row in self.g.query(q):
                e1, e2 = self.EMOTIONMAPPING.get(self._local(row.e1)), self.EMOTIONMAPPING.get(self._local(row.e2))
                if e1 in self.ALLEMOTIONS and e2 in self.ALLEMOTIONS:
                    i, j = self.ALLEMOTIONS.index(e1), self.ALLEMOTIONS.index(e2)
                    mat[i, j] = mat[j, i] = max(mat[i, j], float(row.score))
        except: pass
        s = mat.sum(axis=1, keepdims=True); s[s==0] = 1.0
        return mat / s

    def _build_strict_lexicon(self):
        EK, lookup = self.EKMAN, {}
        mix_cache, sim_cache = {}, {}
        
        # 1. Caching
        for r in self.g.query("PREFIX ekman: <http://example.org/ekman-ontology#> SELECT ?m ?e ?w WHERE { ?m a ekman:EmotionMixture ; ekman:hasComponent ?c . ?c ekman:componentEmotion ?e ; ekman:componentWeight ?w . }"):
            m, e, w = r.m, self.EMOTIONMAPPING.get(self._local(r.e)), float(r.w)
            if e in self.ALLEMOTIONS: mix_cache.setdefault(m, {})[e] = mix_cache.setdefault(m, {}).get(e, 0.0) + w
        
        for s, _, o in self.g.triples((None, EK.hasEmotionCategory, None)):
            e = self.EMOTIONMAPPING.get(self._local(o))
            if e in self.ALLEMOTIONS: sim_cache[s] = [e]
            
        for s in self.g.subjects(RDF.type, None):
            if s in mix_cache or s in sim_cache: continue
            ts = list(self.g.objects(s, RDF.type))
            if EK.NegationCue in ts: sim_cache[s] = ["NegationCue"]
            else:
                for t in ts:
                    m = self.EMOTIONMAPPING.get(self._local(t))
                    if m in self.ALLEMOTIONS: sim_cache[s] = [m]; break
        
        # 2. Build entries
        q = """PREFIX ekman: <http://example.org/ekman-ontology#> SELECT ?word ?stim ?ev ?base ?type ?dom WHERE { ?evoc a ekman:Evocation ; ekman:fromLexiconEntry ?lex ; ekman:toStimulus ?stim . ?lex ekman:hasContent ?word . OPTIONAL { ?evoc ekman:confidenceScore ?ev } OPTIONAL { ?evoc ekman:evidenceType ?type } OPTIONAL { ?lex ekman:baseIntensityScore ?base } OPTIONAL { ?lex ekman:belongsToDomain ?dom } }"""
        for r in self.g.query(q):
            if r.dom and self._local(r.dom) not in self.VALID_DOMAINS: continue 
            w = self._norm(r.word)
            if len(w) < 2 or w in self.AMBIGUOUS_TERMS: continue
            
            emos = {}
            if r.stim in mix_cache:
                raw = mix_cache[r.stim]; tot = sum(raw.values())
                if tot > 0: emos = {k: v/tot for k, v in raw.items()}
            elif r.stim in sim_cache:
                lst = sim_cache[r.stim]
                emos = {k: 1.0 for k in lst}
            if not emos: continue

            # Deep Inference applied here
            apps, ints, pols = self._infer_attributes(r.stim, emos.keys())
            
            entry_score = float(r.ev or self.defaultconf) * float(r.base or 1.0)
            
            entry = {"emotions": emos, "score": entry_score, 
                     "appraisals": list(apps), "intensities": list(ints), "polarities": list(pols), 
                     "isnegation": "NegationCue" in emos}
            lookup.setdefault(w, []).append(entry)
            wu = self._norm(unidecode(w))
            if wu != w: lookup.setdefault(wu, []).append(entry)
        return lookup

    def _infer_attributes(self, s, emos):
        EK, a, i, p = self.EKMAN, set(), set(), set()
        
        # 1. Direct Properties
        def g(r):
            for o in self.g.objects(r, EK.evokesAppraisal): a.add(self._local(o))
            for o in self.g.objects(r, EK.hasIntensity): i.add(self._local(o))
            for o in self.g.objects(r, EK.hasPolarity): p.add(self._local(o))
        g(s)
        for t in self.g.objects(s, RDF.type): g(t)
        
        # 2. Deep Inference (Logic V12.2)
        # Suy diễn Polarity từ Emotion
        if not p:
            for e in emos:
                if e in self.IMPLIED_POLARITY: p.add(self.IMPLIED_POLARITY[e])
        
        # Suy diễn Polarity từ Appraisal (Rất quan trọng cho VSMEC để bắt các sắc thái ẩn)
        if not p:
            for app in a:
                if app in self.IMPLIED_POLARITY: p.add(self.IMPLIED_POLARITY[app])

        return a, i, p

    def getvector(self, text, debug=False):
        tokens = ViTokenizer.tokenize(str(text).lower()).split()
        flat = [self._norm(t) for t in tokens if self._norm(t)]
        
        ve, va, vi = np.zeros(7, dtype=np.float32), np.zeros(9, dtype=np.float32), np.zeros(3, dtype=np.float32)
        vp, vn = np.zeros(3, dtype=np.float32), np.zeros(2, dtype=np.float32)
        
        idx, n = 0, len(flat)
        hit_trace = []

        while idx < n:
            m, ml = None, 0
            for L in range(min(self.mwe_max_len, n-idx), 0, -1):
                p = " ".join(flat[idx:idx+L])
                if p in self.lexiconmap: m, ml = self.lexiconmap[p], L; break
            
            if not m:
                idx += 1; continue
            
            if debug: hit_trace.append({"phrase": " ".join(flat[idx:idx+ml]), "data": m[0]})

            for ent in m:
                if ent["isnegation"]:
                    vn[0] = 1.0; continue
                
                sc = ent["score"]
                # Soft Negation: Chỉ giảm điểm, không đảo cực (No Flip)
                if vn[0] > 0: sc *= self.negation_attenuation 
                
                for e, w in ent["emotions"].items():
                    if e in self.ALLEMOTIONS: ve[self.ALLEMOTIONS.index(e)] += sc * w
                for x in ent["appraisals"]:
                    if x in self.ALLAPPRAISALS: va[self.ALLAPPRAISALS.index(x)] += sc
                for x in ent["intensities"]:
                    if x in self.INTENSITY_CLASSES: vi[self.INTENSITY_CLASSES.index(x)] = 1.0
                
                # Soft Polarity: Cộng dồn theo Score
                for x in ent["polarities"]:
                    if x in self.POLARITY_CLASSES: vp[self.POLARITY_CLASSES.index(x)] += sc 
            idx += ml

        if self.alpha_similarity > 0: ve = np.clip(ve + self.alpha_similarity * (ve @ self.sim_matrix), 0, None)
        fin = np.concatenate([np.tanh(ve), np.tanh(va), vi, vp, vn]).astype(np.float32)
        return (fin, flat, hit_trace) if debug else fin

# KHỞI TẠO (Dùng file ekman6_4_8 như bạn yêu cầu)
RDFPATH = "/kaggle/input/kg-ekman6/ekman6_4_8.rdf"
if os.path.exists(RDFPATH):
    print(f"🚀 Initializing OntologyEngine v12.2 for VSMEC (Deep Inference) with {RDFPATH}...")
    ontengine = OntologyEngine(RDFPATH, verbose=True)
else:
    print(f"❌ ERROR: File not found {RDFPATH}")

🚀 Initializing OntologyEngine v12.2 for VSMEC (Deep Inference) with /kaggle/input/kg-ekman6/ekman6_4_8.rdf...
📥 Loading KG for VSMEC: /kaggle/input/kg-ekman6/ekman6_4_8.rdf
   ✅ Loaded 90420 triples.
   📚 Lexicon size: 2540 entries (v12.2 VSMEC-Ready)


In [4]:
# Thử test câu này để ép code chạy Soft Negation
t_test = "không thực sự thích" 
v, _, tr = ontengine.getvector(t_test, debug=True)

print(f"Matches: {[x['phrase'] for x in tr]}") 
# Nếu kết quả ra: ['không', 'thích'] -> Soft Negation đã chạy!
print(f"Negation Flag: {v[-2]}") # Sẽ ra 1.0

Matches: ['không', 'thích']
Negation Flag: 1.0


In [5]:
# =========================================================
# CELL: DIAGNOSTIC TEST SUITE CHO KG ENGINE V11.0
# Mục tiêu: Kiểm tra tự động 4 tính năng: 
# 1. Soft Negation (Giảm điểm) 
# 2. Polarity Flip (Đảo cực tính) 
# 3. Blacklist (Lọc nhiễu) 
# 4. Scope Fix (Xuyên từ rác)
# =========================================================
import pandas as pd
import numpy as np

# --- 1. HÀM GIẢI MÃ VECTOR (Vector -> Text) ---
def decode_vector(vec, engine):
    # 7 chiều đầu: Emotions
    emo_vals = vec[:7]
    if np.max(emo_vals) > 0.01:
        top_idx = np.argmax(emo_vals)
        emotion = engine.ALLEMOTIONS[top_idx]
        score = emo_vals[top_idx]
    else:
        emotion = "Neutral/None"
        score = 0.0
        
    # Index 19, 20, 21: Polarity [Pos, Neg, Neu]
    pol_vals = vec[19:22]
    pol_labels = ["Positive", "Negative", "Neutral"]
    if np.max(pol_vals) > 0:
        polarity = pol_labels[np.argmax(pol_vals)]
    else:
        polarity = "Unknown" # Nếu vẫn ra Unknown tức là KG chưa gán Polarity cho từ đó
        
    has_neg = vec[23] > 0
    return emotion, score, polarity, has_neg
    
# --- 2. ĐỊNH NGHĨA TEST CASES ---
# Format: (Input Text, Kỳ vọng Emotion, Kỳ vọng Polarity, Mô tả lỗi cần check)
test_cases = [
    # --- NHÓM 1: CƠ BẢN (Sanity Check) ---
    ("món này ngon tuyệt", "Happiness", "Positive", "Basic Positive"),
    ("thất vọng tràn trề", "Sadness", "Negative", "Basic Negative"),

    # --- NHÓM 2: SOFT NEGATION (Quan trọng nhất) ---
    # v11.0: Emotion phải giảm điểm (về gần 0 hoặc Neutral), Polarity phải là Negative
    ("tôi không thích nó", "Neutral/None", "Negative", "Soft Negation (Không thích)"),
    ("chẳng vui chút nào", "Neutral/None", "Negative", "Soft Negation (Chẳng vui)"),

    # --- NHÓM 3: SCOPE FIX (Lỗi 'hề') ---
    # Phải bắt được phủ định dù có từ đệm
    ("tôi không hề thích", "Neutral/None", "Negative", "Scope Skip (Không hề...)"),
    ("không phải là ghét", "Neutral/None", "Positive", "Double Negation (Không ghét -> Pos)"),

    # --- NHÓM 4: BLACKLIST (Lỗi từ đa nghĩa) ---
    # Không được bắt chữ "hay" (hoặc) và "chả" (món ăn)
    ("làm hay nghỉ", "Neutral/None", "Unknown", "Blacklist: Hay (Or)"),
    ("bún chả rất ngon", "Happiness", "Positive", "Blacklist: Chả (Food)"),
    ("tôi là sinh viên", "Neutral/None", "Unknown", "Blacklist: Tôi/Là (Pronouns)")
]

# --- 3. CHẠY TEST TỰ ĐỘNG ---
print(f"{'='*25} BÁO CÁO KIỂM THỬ KG V11.0 {'='*25}")
results = []
pass_count = 0

for text, exp_emo, exp_pol, desc in test_cases:
    # Chạy Engine
    vec, _, trace = ontengine.getvector(text, debug=True)
    out_emo, out_score, out_pol, out_neg = decode_vector(vec, ontengine)
    
    # LOGIC CHẤM ĐIỂM (PASS/FAIL)
    status = "❌ FAIL"
    note = ""
    
    # Check 1: Emotion
    # Nếu kỳ vọng là Neutral/None -> Score thực tế phải thấp (< 0.5) hoặc label là Neutral
    emo_pass = False
    if exp_emo == "Neutral/None":
        if out_score < 0.5 or out_emo == "Neutral": emo_pass = True
    else:
        if out_emo == exp_emo and out_score > 0.3: emo_pass = True # Score > 0.3 là active
        
    # Check 2: Polarity (Rất quan trọng cho Sentiment 3 nhãn)
    pol_pass = False
    if exp_pol == "Unknown":
        pol_pass = True # Không care
    elif out_pol == exp_pol:
        pol_pass = True
        
    # Đặc biệt: Case "bún chả" -> Fail nếu bắt nhầm negation
    if "Blacklist: Chả" in desc and out_neg:
        status = "❌ FAIL (False Negation)"
    # Đặc biệt: Case "không thích" -> Fail nếu vẫn là Positive
    elif "Soft Negation" in desc and out_pol == "Positive":
        status = "❌ FAIL (Wrong Polarity)"
    elif emo_pass and pol_pass:
        status = "✅ PASS"
        pass_count += 1
    
    # Log kết quả
    matched_str = ", ".join([t['phrase'] for t in trace])
    results.append({
        "Scenario": desc,
        "Input": text,
        "Matched": matched_str,
        "Prediction": f"{out_emo} ({out_score:.2f}) | {out_pol}",
        "NegFlag": "YES" if out_neg else "NO",
        "Status": status
    })

# --- 4. HIỂN THỊ KẾT QUẢ ---
df_res = pd.DataFrame(results)
# Chỉ hiển thị các cột quan trọng
print(df_res[["Input", "Matched", "Prediction", "Status"]].to_string(index=False))

print("-" * 70)
print(f"🏆 KẾT QUẢ: {pass_count}/{len(test_cases)} Test Cases Passed.")

if pass_count == len(test_cases):
    print("🚀 KẾT LUẬN: KG Engine v11.0 hoạt động HOÀN HẢO. Sẵn sàng train!")
else:
    print("⚠️ KẾT LUẬN: Vẫn còn lỗi (xem các dòng FAIL).")

========================= BÁO CÁO KIỂM THỬ KG V11.0 =========================
             Input           Matched                    Prediction                  Status
món này ngon tuyệt             tuyệt   Happiness (0.75) | Positive                  ✅ PASS
thất vọng tràn trề         thất vọng     Sadness (0.58) | Negative                  ✅ PASS
tôi không thích nó       không thích     Disgust (0.55) | Negative                  ❌ FAIL
chẳng vui chút nào               vui   Happiness (0.79) | Positive ❌ FAIL (Wrong Polarity)
tôi không hề thích      không, thích   Happiness (0.45) | Positive                  ❌ FAIL
không phải là ghét không, phải, ghét       Anger (0.17) | Negative                  ❌ FAIL
      làm hay nghỉ              nghỉ     Sadness (0.39) | Negative                  ✅ PASS
  bún chả rất ngon               rất   Happiness (0.42) | Positive                  ✅ PASS
  tôi là sinh viên                   Neutral/None (0.00) | Unknown                  ✅ PASS
------------

In [6]:
# ---------------------------------------------------------
# CELL 3: DATASET & PREPROCESSING (VSMEC)
# ---------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, engine, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.engine = engine
        self.max_len = max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        txt = str(self.df.iloc[idx]['Sentence'])
        lbl = self.df.iloc[idx]['label_idx']
        enc = self.tokenizer.encode_plus(txt, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        vec = self.engine.getvector(txt)
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(), 'ontology_features': torch.tensor(vec, dtype=torch.float), 'labels': torch.tensor(lbl, dtype=torch.long)}

try:
    # VSMEC Path
    base = "/kaggle/input/uit-vsmec"
    df_train = pd.read_excel(f"{base}/train_nor_811.xlsx")
    df_valid = pd.read_excel(f"{base}/valid_nor_811.xlsx")
    df_test = pd.read_excel(f"{base}/test_nor_811.xlsx")
    
    # Đổi tên cột cho đồng nhất
    for df in [df_train, df_valid, df_test]:
        df.dropna(subset=['Sentence', 'Emotion'], inplace=True)
        # Map 'Other' thành 'Other' (hoặc bạn có thể map thành Neutral nếu muốn)
    print("✅ VSMEC Data Loaded")
except Exception as e: 
    print(f"❌ Load Error: {e}")

# Label Encoding
labels = sorted(df_train['Emotion'].unique().astype(str))
le = LabelEncoder()
le.fit(labels)
print(f"📋 Classes: {le.classes_}")

for df in [df_train, df_valid, df_test]:
    df['label_idx'] = le.transform(df['Emotion'].astype(str))

train_loader = DataLoader(EmotionDataset(df_train, tokenizer, ontengine), batch_size=32, shuffle=True)
valid_loader = DataLoader(EmotionDataset(df_valid, tokenizer, ontengine), batch_size=32)
test_loader = DataLoader(EmotionDataset(df_test, tokenizer, ontengine), batch_size=32)

# Class Weights
cls_w = compute_class_weight('balanced', classes=np.unique(df_train['label_idx']), y=df_train['label_idx'])
weights_tensor = torch.tensor(cls_w, dtype=torch.float).to(device)
print(f"⚖️ Weights: {weights_tensor}")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ VSMEC Data Loaded
📋 Classes: ['Anger' 'Disgust' 'Enjoyment' 'Fear' 'Other' 'Sadness' 'Surprise']
⚖️ Weights: tensor([2.0270, 0.7400, 0.5087, 2.4924, 0.7763, 0.8369, 3.2751],
       device='cuda:0')


In [7]:
# ---------------------------------------------------------
# CELL 4: MODEL (xAI & DENSE SUPPORT)
# ---------------------------------------------------------
try: ONT_INPUT_DIM = len(ontengine.getvector("t"))
except: ONT_INPUT_DIM = 24

class PhoBERT_Fusion_V2(nn.Module):
    def __init__(self, n_classes, fusion_type='none', ontology_dim=None, ont_input_dim=ONT_INPUT_DIM):
        super(PhoBERT_Fusion_V2, self).__init__()
        self.fusion_type = fusion_type
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
        self.dropout = nn.Dropout(p=0.3)
        
        # Ontology Adapter
        if ontology_dim: # Dense Mode (64d)
            self.ont_adapter = nn.Sequential(
                nn.Linear(ont_input_dim, ontology_dim),
                nn.BatchNorm1d(ontology_dim),
                nn.ReLU(),
                nn.Dropout(0.1)
            )
            self.curr_dim = ontology_dim
        else: # Raw Mode (xAI)
            self.ont_adapter = nn.Identity()
            self.curr_dim = ont_input_dim
            
        # Fusion
        if self.fusion_type == 'gate':
            self.gate = nn.Linear(768, self.curr_dim)
            
        # Head
        inp = 768 + self.curr_dim if self.fusion_type != 'none' else 768
        self.fc = nn.Linear(inp, n_classes)

    def forward(self, ids, mask, ont):
        _, txt = self.phobert(ids, mask, return_dict=False)
        txt = self.dropout(txt)
        
        if self.fusion_type == 'none': return self.fc(txt)
        
        ont_emb = self.ont_adapter(ont)
        if self.fusion_type == 'gate':
            g = torch.sigmoid(self.gate(txt))
            ont_emb = ont_emb * g
            
        comb = torch.cat([txt, ont_emb], dim=1)
        return self.fc(comb)

In [8]:
# ---------------------------------------------------------
# CELL 5: TRAINER
# ---------------------------------------------------------
def train_experiment(model, name, epochs=15, patience=4):
    print(f"\n🚀 START: {name}")
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
    model.to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=2e-5)
    crit = nn.CrossEntropyLoss(weight=weights_tensor)
    
    best_f1, pat = 0, 0
    best_w = copy.deepcopy(model.state_dict()) if not isinstance(model, nn.DataParallel) else copy.deepcopy(model.module.state_dict())
    
    for ep in range(epochs):
        model.train()
        loss_acc = 0
        for b in train_loader:
            optim.zero_grad()
            ids, m, o, l = [b[k].to(device) for k in ['input_ids','attention_mask','ontology_features','labels']]
            loss = crit(model(ids, m, o), l)
            loss.backward(); optim.step()
            loss_acc += loss.item()
            
        acc, f1 = evaluate(model, valid_loader)
        print(f"Ep {ep+1} | Loss: {loss_acc/len(train_loader):.4f} | Val F1: {f1:.4f}", end=" ")
        
        if f1 > best_f1:
            best_f1 = f1; pat = 0; best_w = copy.deepcopy(model.state_dict()) if not isinstance(model, nn.DataParallel) else copy.deepcopy(model.module.state_dict())
            print("✅")
        else:
            pat += 1
            print(f"⚠️ {pat}")
        if pat >= patience: break
    
    if isinstance(model, nn.DataParallel): model.module.load_state_dict(best_w)
    else: model.load_state_dict(best_w)
    return model

def evaluate(model, loader):
    model.eval()
    p, t = [], []
    with torch.no_grad():
        for b in loader:
            ids, m, o, l = [b[k].to(device) for k in ['input_ids','attention_mask','ontology_features','labels']]
            _, pr = torch.max(model(ids, m, o), 1)
            p.extend(pr.cpu().numpy()); t.extend(l.cpu().numpy())
    return accuracy_score(t, p), f1_score(t, p, average='macro')

In [ ]:
# ---------------------------------------------------------
# CELL 6: MAIN EXPERIMENTS (5 CASES - VSMEC)
# ---------------------------------------------------------
NC = len(le.classes_)

# 1. Baseline
m1 = PhoBERT_Fusion_V2(NC, 'none')
m1 = train_experiment(m1, "Baseline (PhoBERT)", epochs=15)



2026-01-08 04:20:11.921825: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767846012.225004      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767846012.320739      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767846013.165159      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767846013.165207      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767846013.165209      55 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 START: Baseline (PhoBERT)


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Ep 1 | Loss: 1.6746 | Val F1: 0.4048 ✅
Ep 2 | Loss: 1.1193 | Val F1: 0.4857 ✅
Ep 3 | Loss: 0.8445 | Val F1: 0.5577 ✅
Ep 4 | Loss: 0.6420 | Val F1: 0.5461 ⚠️ 1
Ep 5 | Loss: 0.4939 | Val F1: 0.5666 ✅
Ep 6 | Loss: 0.3713 | Val F1: 0.5663 ⚠️ 1
Ep 7 | Loss: 0.2836 | Val F1: 0.5518 ⚠️ 2
Ep 8 | Loss: 0.2161 | Val F1: 0.5796 ✅
Ep 9 | Loss: 0.1665 | Val F1: 0.5809 ✅
Ep 10 | Loss: 0.1523 | Val F1: 0.5281 ⚠️ 1


In [ ]:
# 2. Raw + Concat (xAI)
m2 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=None)
m2 = train_experiment(m2, "Raw + Concat", epochs=15)

# 3. Raw + Gate
m3 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=None)
m3 = train_experiment(m3, "Raw + Gate", epochs=15)



In [ ]:
# 4. Dense (64d) + Concat
m4 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=64)
m4 = train_experiment(m4, "Dense(64) + Concat", epochs=15)

# 5. Dense (64d) + Gate
m5 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=64)
m5 = train_experiment(m5, "Dense(64) + Gate", epochs=15)

In [ ]:
# ---------------------------------------------------------
# CELL 7: FINAL REPORT
# ---------------------------------------------------------
def gp(model, loader):
    model.eval()
    p, t = [], []
    with torch.no_grad():
        for b in loader:
            ids, m, o, l = [b[k].to(device) for k in ['input_ids','attention_mask','ontology_features','labels']]
            _, pr = torch.max(model(ids, m, o), 1)
            p.extend(pr.cpu().numpy()); t.extend(l.cpu().numpy())
    return t, p

y_true, p1 = gp(m1, test_loader)
_, p2 = gp(m2, test_loader)
_, p3 = gp(m3, test_loader)
_, p4 = gp(m4, test_loader)
_, p5 = gp(m5, test_loader)

res = {
    "Model": ["Baseline", "Raw+Concat", "Raw+Gate", "Dense+Concat", "Dense+Gate"],
    "F1-Macro": [f1_score(y_true, p, average='macro') for p in [p1, p2, p3, p4, p5]],
    "Accuracy": [accuracy_score(y_true, p) for p in [p1, p2, p3, p4, p5]]
}

print("\n" + "="*50)
print("🏆 VSMEC FINAL RESULTS")
print("="*50)
print(pd.DataFrame(res).round(4).to_markdown(index=False))

print("\n" + "="*50)
print("📑 DETAILED REPORT (DENSE + GATE)")
print("="*50)
print(classification_report(y_true, p5, target_names=le.classes_, digits=4))

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
import sys

# ---------------------------------------------------------
# 1. ĐỊNH NGHĨA MODEL: DEEP ONTOLOGY FUSION
# ---------------------------------------------------------

# Tự động lấy số lượng class (nếu có biến le từ các cell trước)
try:
    NUM_CLASSES = len(le.classes_)
except NameError:
    NUM_CLASSES = 7  # Mặc định cho VSFC

# Tự động lấy kích thước vector ontology (nếu có từ cell trước)
try:
    ONT_DIM = ONT_INPUT_DIM 
except NameError:
    ONT_DIM = 24 # Mặc định

class PhoBERT_DeepOntology(nn.Module):
    def __init__(self, n_classes, fusion_type='concat', ont_input_dim=24, hidden_dim=64):
        super(PhoBERT_DeepOntology, self).__init__()
        self.fusion_type = fusion_type
        
        # Backbone PhoBERT (768d)
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
        self.dropout = nn.Dropout(p=0.3)
        
        # --- KHỐI XỬ LÝ ONTOLOGY (Deep Projection) ---
        # Cấu trúc: Input -> Hidden (ReLU) -> Output (768)
        self.ont_pipeline = nn.Sequential(
            nn.Linear(ont_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            # Phóng chiếu sang không gian 768 chiều để khớp với PhoBERT
            nn.Linear(hidden_dim, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Cổng Fusion (Gate)
        if self.fusion_type == 'gate':
            self.gate_fc = nn.Linear(768 * 2, 768) 
        
        # Classifier Head
        final_input_dim = 768 * 2 if self.fusion_type == 'concat' else 768
        self.fc = nn.Linear(final_input_dim, n_classes)

    def forward(self, input_ids, attention_mask, ontology_features):
        # 1. PhoBERT Forward
        _, text_emb = self.phobert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        text_emb = self.dropout(text_emb)
        
        # 2. Ontology Forward
        ont_emb = self.ont_pipeline(ontology_features)
        
        # 3. Fusion Strategy
        if self.fusion_type == 'concat':
            combined = torch.cat([text_emb, ont_emb], dim=1)
            return self.fc(combined)
            
        elif self.fusion_type == 'gate':
            concat_feat = torch.cat([text_emb, ont_emb], dim=1)
            z = torch.sigmoid(self.gate_fc(concat_feat))
            fused_emb = (z * text_emb) + ((1 - z) * ont_emb)
            return self.fc(fused_emb)
            
        else: # 'none' hoặc 'add'
            return self.fc(text_emb + ont_emb)

# ---------------------------------------------------------
# 2. THỰC NGHIỆM BỔ SUNG: WIDE DEEP ONTOLOGY (24 -> 768 -> 768)
# ---------------------------------------------------------

# === CASE C: WIDE DEEP (768d) + CONCAT ===
print("\n" + "="*40)
print("🧪 CASE C: Wide Deep (24->768->768) + CONCAT")
print("="*40)

# Khởi tạo model với hidden_dim = 768 (Wide) thay vì 64 (Bottleneck)
model_wide_concat = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='concat', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=768 
)

# Kiểm tra cấu trúc layer đầu tiên để chắc chắn nó là 24->768
print(f"🧐 Cấu trúc lớp ẩn: {model_wide_concat.ont_pipeline[0]}")

# Train
model_wide_concat = train_experiment(
    model_wide_concat, 
    "Wide Ont (768d) + Concat", 
    epochs=15, 
    patience=4
)
acc_c, f1_c = evaluate(model_wide_concat, test_loader)


# === CASE D: WIDE DEEP (768d) + GATE ===
print("\n" + "="*40)
print("🧪 CASE D: Wide Deep (24->768->768) + GATE")
print("="*40)

model_wide_gate = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='gate', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=768
)

# Train
model_wide_gate = train_experiment(
    model_wide_gate, 
    "Wide Ont (768d) + Gate", 
    epochs=15, 
    patience=4
)
acc_d, f1_d = evaluate(model_wide_gate, test_loader)

# === TỔNG HỢP KẾT QUẢ MỚI ===
print(f"\n📊 KẾT QUẢ KIẾN TRÚC WIDE (24->768->768):")
print(f"1. Concat: F1-Macro = {f1_c:.4f} | Acc = {acc_c:.4f}")
print(f"2. Gate  : F1-Macro = {f1_d:.4f} | Acc = {acc_d:.4f}")

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

# ---------------------------------------------------------
# 1. ĐỊNH NGHĨA CLASS MODEL (Hỗ trợ tùy chỉnh hidden_dim)
# ---------------------------------------------------------

# Tự động lấy tham số môi trường
try:
    NUM_CLASSES = len(le.classes_)
    ONT_DIM = ONT_INPUT_DIM 
except NameError:
    NUM_CLASSES = 7
    ONT_DIM = 24 

class PhoBERT_DeepOntology(nn.Module):
    def __init__(self, n_classes, fusion_type='concat', ont_input_dim=24, hidden_dim=64):
        super(PhoBERT_DeepOntology, self).__init__()
        self.fusion_type = fusion_type
        
        # Backbone PhoBERT
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
        self.dropout = nn.Dropout(p=0.3)
        
        # --- KHỐI DEEP ONTOLOGY ---
        self.ont_pipeline = nn.Sequential(
            # Layer 1: Nén (Compression) -> Bottleneck 64
            nn.Linear(ont_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            # Layer 2: Phóng chiếu (Projection) -> 768 (Khớp với PhoBERT)
            nn.Linear(hidden_dim, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Cổng Fusion (Gate)
        if self.fusion_type == 'gate':
            self.gate_fc = nn.Linear(768 * 2, 768) 
        
        # Classifier Head
        final_input_dim = 768 * 2 if self.fusion_type == 'concat' else 768
        self.fc = nn.Linear(final_input_dim, n_classes)

    def forward(self, input_ids, attention_mask, ontology_features):
        # 1. Text Embedding
        _, text_emb = self.phobert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        text_emb = self.dropout(text_emb)
        
        # 2. Ontology Embedding (24 -> 64 -> 768)
        ont_emb = self.ont_pipeline(ontology_features)
        
        # 3. Fusion
        if self.fusion_type == 'concat':
            combined = torch.cat([text_emb, ont_emb], dim=1)
            return self.fc(combined)
        elif self.fusion_type == 'gate':
            concat_feat = torch.cat([text_emb, ont_emb], dim=1)
            z = torch.sigmoid(self.gate_fc(concat_feat))
            fused_emb = (z * text_emb) + ((1 - z) * ont_emb)
            return self.fc(fused_emb)
        else:
            return self.fc(text_emb + ont_emb)

# ---------------------------------------------------------
# 2. CHẠY THỰC NGHIỆM: CASE BOTTLENECK (64d)
# ---------------------------------------------------------

# === CASE A: BOTTLENECK (64d) + CONCAT ===
print("\n" + "="*50)
print("🧪 CASE A: Bottleneck Deep (24->64->768) + CONCAT")
print("="*50)

# Khởi tạo model với hidden_dim = 64
model_bottleneck_concat = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='concat', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=64  # <--- ĐIỂM QUAN TRỌNG: Nén xuống 64
)

# Kiểm tra in ra cấu trúc để chắc chắn
print(f"🧐 Cấu trúc lớp ẩn: {model_bottleneck_concat.ont_pipeline[0]}") 
# Mong đợi: Linear(in_features=24, out_features=64, bias=True)

model_bottleneck_concat = train_experiment(
    model_bottleneck_concat, 
    "Bottleneck (64d) + Concat", 
    epochs=15, 
    patience=4
)
acc_a, f1_a = evaluate(model_bottleneck_concat, test_loader)


# === CASE B: BOTTLENECK (64d) + GATE ===
print("\n" + "="*50)
print("🧪 CASE B: Bottleneck Deep (24->64->768) + GATE")
print("="*50)

model_bottleneck_gate = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='gate', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=64  # <--- Nén xuống 64
)

model_bottleneck_gate = train_experiment(
    model_bottleneck_gate, 
    "Bottleneck (64d) + Gate", 
    epochs=15, 
    patience=4
)
acc_b, f1_b = evaluate(model_bottleneck_gate, test_loader)

# === TỔNG HỢP KẾT QUẢ ===
print(f"\n📊 KẾT QUẢ KIẾN TRÚC BOTTLENECK (24->64->768):")
print(f"1. Concat: F1-Macro = {f1_a:.4f} | Acc = {acc_a:.4f}")
print(f"2. Gate  : F1-Macro = {f1_b:.4f} | Acc = {acc_b:.4f}")

In [ ]:
# =========================================================
# CELL: FINAL EVALUATION FOR VSMEC (7 LABELS)
# Fix: Cập nhật LABEL_NAMES cho đủ 7 cảm xúc
# =========================================================
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, f1_score, accuracy_score

# 1. CẤU HÌNH NHÃN CHO VSMEC (7 Lớp)
# Lưu ý: Thứ tự này phải khớp với Label Encoder lúc train.
# Thứ tự phổ biến của VSMEC thường là:
LABEL_NAMES = ["Enjoyment", "Sadness", "Fear", "Anger", "Disgust", "Surprise", "Other"]
# Nếu kết quả bị lệch nhãn, hãy kiểm tra lại dataset.classes hoặc le.classes_

def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            ont = batch['ontology_features'].to(device)
            lbl = batch['labels'].to(device)
            
            # Forward
            outputs = model(ids, mask, ont)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(lbl.cpu().numpy())
    return all_labels, all_preds

# --- 2. TỰ ĐỘNG QUÉT CÁC MODEL VSMEC TRONG RAM ---
models_to_test = []

# Baseline
if 'm1' in globals(): models_to_test.append(('Baseline', m1))
# Shallow Fusion
if 'm2' in globals(): models_to_test.append(('Raw+Concat', m2))
if 'm3' in globals(): models_to_test.append(('Raw+Gate', m3))
# Deep Fusion (Nếu bạn chạy các case sau)
if 'model_deep_concat' in globals(): models_to_test.append(('Deep+Concat', model_deep_concat))
if 'model_deep_gate' in globals(): models_to_test.append(('Deep+Gate', model_deep_gate))

# --- 3. BÁO CÁO F1-SCORE ---
print(f"\n{'='*40}\n📊 BÁO CÁO HIỆU NĂNG (VSMEC - 7 LABELS)\n{'='*40}")

reports = {}
best_model_obj = None
best_model_name = ""
best_f1 = -1

for name, model in models_to_test:
    print(f"\n🔹 MODEL: {name}")
    y_true, y_pred = get_predictions(model, test_loader)
    
    # In báo cáo chi tiết
    try:
        print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))
    except ValueError:
        # Fallback nếu số lượng class thực tế không khớp 7
        print("⚠️ Cảnh báo: Số lượng class dự đoán không khớp danh sách tên.")
        print(classification_report(y_true, y_pred, digits=4))

    f1 = f1_score(y_true, y_pred, average='macro')
    acc = accuracy_score(y_true, y_pred)
    
    reports[name] = {"Accuracy": acc, "F1-Macro": f1}
    
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_model_obj = model

if len(reports) > 0:
    print(f"\n{'='*40}\n🏆 BẢNG XẾP HẠNG F1-MACRO\n{'='*40}")
    df_res = pd.DataFrame(reports).T
    print(df_res.sort_values(by="F1-Macro", ascending=False))
    print(f"\n🌟 Nhà vô địch: {best_model_name} (F1: {best_f1:.4f})")

# =========================================================
# 4. SO KÈO THỰC TẾ (INFERENCE BATTLE)
# Test khả năng sửa sai của Ontology trên các câu khó của VSMEC
# =========================================================
def predict_emotion(text, model):
    inputs = tokenizer(text, return_tensors='pt', padding='max_length', truncation=True, max_length=128)
    for k, v in inputs.items(): inputs[k] = v.to(device)
    
    v = ontengine.getvector(text)
    ont_tensor = torch.tensor([vec], dtype=torch.float).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(inputs['input_ids'], inputs['attention_mask'], ont_tensor)
        probs = torch.softmax(outputs, dim=1)
        conf, pred_idx = torch.max(probs, 1)
        
    return LABEL_NAMES[pred_idx.item()], conf.item()

# Các câu test VSMEC (Cảm xúc đa dạng)
hard_cases = [
    "tôi không thích thái độ đó",     # Disgust/Anger (Soft Negation)
    "buồn cười thật sự",              # Enjoyment (Ambiguity 'buồn')
    "sợ muốn thót tim",               # Fear
    "chả có gì đặc biệt",             # Other/Neutral (Blacklist 'chả')
    "bất ngờ chưa bà già",            # Surprise
    "cũng tạm ổn thôi"                # Other/Neutral
]

# Chọn Baseline và Model tốt nhất để so sánh
baseline_model = globals().get("m1", None)
champion_model = best_model_obj

print(f"\n{'='*70}\n⚔️ SO KÈO: BASELINE vs {best_model_name.upper()}\n{'='*70}")

if baseline_model and champion_model:
    print(f"{'Input Text':<30} | {'Baseline':<15} | {'Champion':<15} | {'KG Hint (Emo)'}")
    print("-" * 100)
    
    for text in hard_cases:
        base_lbl, base_conf = predict_emotion(text, baseline_model)
        champ_lbl, champ_conf = predict_emotion(text, champion_model)
        
        # KG Hint cho VSMEC (7 classes)
        # Vector 24 chiều: [0-6] là Emotion Score
        v, _, _ = ontengine.getvector(text)
        emo_vals = v[:7]
        kg_hint = "Neutral"
        if np.max(emo_vals) > 0.01:
            # Map index sang tên emotion
            # Lưu ý: check thứ tự trong ontengine.ALLEMOTIONS
            kg_hint = ontengine.ALLEMOTIONS[np.argmax(emo_vals)]
            
        if v[23] > 0: kg_hint += " (+NEG)"
        
        print(f"'{text[:28]:<28}' | {base_lbl} ({base_conf:.2f}) | {champ_lbl} ({champ_conf:.2f})   | {kg_hint}")
else:
    print("⚠️ Cần có ít nhất Baseline và 1 Model Ontology trong RAM.")